In [3]:

from graph.workflow import review_graph


filename = "auth.js"

diff = """
+app.get("/user", (req, res) => {
+    const userId = req.query.id;
+    const query = "SELECT * FROM users WHERE id = " + userId;
+    db.execute(query, (err, result) => {
+        res.json(result);
+    });
+});
"""


result = review_graph.invoke({
    "filename": filename,
    "diff": diff,
    "review": None,
    "error": None,
    "status": "pending"
})


print("\nWorkflow Status:")
print(result["status"])

print("\nReview Result:")
print(result["review"])

print("\nError:")
print(result["error"])



Workflow Status:
completed

Review Result:
{'summary': 'The added /user route in auth.js contains a critical SQL injection vulnerability, lacks database error handling, and exposes all user table columns without input validation.', 'findings': [{'severity': 'critical', 'category': 'security', 'file': 'auth.js', 'line': None, 'title': 'SQL injection via string concatenation', 'description': 'User-supplied input from req.query.id is concatenated directly into the SQL query string before passing it to db.execute, allowing attackers to manipulate the query.', 'suggested_fix': 'Use a parameterized query instead, such as db.execute("SELECT id, username, email FROM users WHERE id = ?", [userId], (err, result) => { ... }).'}, {'severity': 'medium', 'category': 'bug', 'file': 'auth.js', 'line': None, 'title': 'Unhandled database error in callback', 'description': 'The err parameter in the db.execute callback is never checked. If the query fails, the endpoint will still send a 200 OK response w

In [4]:
result = review_graph.invoke({
    "filename": "",
    "diff": "",
    "review": None,
    "error": None,
    "status": "pending"
})

print(result)

{'filename': '', 'diff': '', 'review': None, 'error': 'Filename and code difference is required', 'status': 'failed'}


In [5]:
print(review_graph.get_graph().draw_ascii())

                            +-----------+                   
                            | __start__ |                   
                            +-----------+                   
                                  *                         
                                  *                         
                                  *                         
                         +----------------+                 
                         | Validate Input |                 
                         +----------------+.                
                           ...              .....           
                          .                      ...        
                        ..                          .....   
                  +--------+                             .. 
                  | Review |                              . 
                  +--------+.                             . 
               ...           ..                           . 
              .         

In [6]:
result = review_graph.invoke({
    "filename": "auth.js",
    "diff": """
+const query = "SELECT * FROM users WHERE id = " + userId;
+db.execute(query, callback);
""",
    "review": None,
    "error": None,
    "status": "pending"
})

print("Status:", result["status"])
print("Error:", result["error"])
print("Review:", result["review"])

Status: completed
Error: None
Review: {'summary': 'SQL injection vulnerability identified in auth.js due to direct string concatenation in a database query.', 'findings': [{'severity': 'critical', 'category': 'security', 'file': 'auth.js', 'line': None, 'title': 'SQL injection via string concatenation', 'description': 'The userId variable is concatenated directly into the SQL query string before passing it to db.execute(). If userId originates from untrusted input, an attacker can manipulate the query structure to read or modify unauthorized database records.', 'suggested_fix': "Use a parameterized query with placeholders instead of string concatenation, for example: db.execute('SELECT * FROM users WHERE id = ?', [userId], callback);"}]}


In [8]:
from graph.nodes import validate_review

invalid_review = {
    "summary": "Some issues found",
    "findings": [
        {
            "severity": "extreme",
            "category": "security",
            "file": "auth.js",
            "line": None,
            "title": "Invalid severity",
            "description": "Testing validation",
            "suggested_fix": "Fix the issue"
        }
    ]
}

result = validate_review({
    "review": invalid_review
})

print(result)

{'status': 'Completed'}


In [9]:
from graph.nodes import validate_review

invalid_review = {
    "summary": "Test review",
    "findings": [
        {
            "severity": "extreme",
            "category": "security",
            "file": "auth.js",
            "line": None,
            "title": "Test issue",
            "description": "Testing invalid severity",
            "suggested_fix": "Fix it"
        }
    ]
}

print(validate_review({"review": invalid_review}))

{'status': 'Completed'}


In [10]:
import graph.nodes as nodes
import inspect

print("Loaded file:", nodes.__file__)
print(inspect.getsource(nodes.validate_review))

Loaded file: c:\Users\amrit\OneDrive\Desktop\Course Project\PR-Sentinel\ai-engine\graph\nodes.py

def validate_review(state):
    review=state.get("review")

    if not isinstance(review,dict):
        return {
            "status":"Failed",
            "error":"Review output must be a dictionary"
        }

    try:
        validate_review=CodeReview.model_validate(review)

        return {
            "review":validate_review.model_dump(),
            "status":"completed",
            "error":None
        }

    except ValidationError as e:
        return{
            "status":"failed",
            "error":f"Invalid review output: {e}"
        }

    # return {
    #     "status":"Completed"
    # }



In [11]:
import importlib
import graph.nodes as nodes

importlib.reload(nodes)

print(inspect.getsource(nodes.validate_review))

def validate_review(state):
    review=state.get("review")

    if not isinstance(review,dict):
        return {
            "status":"Failed",
            "error":"Review output must be a dictionary"
        }

    try:
        validate_review=CodeReview.model_validate(review)

        return {
            "review":validate_review.model_dump(),
            "status":"completed",
            "error":None
        }

    except ValidationError as e:
        return{
            "status":"failed",
            "error":f"Invalid review output: {e}"
        }

    # return {
    #     "status":"Completed"
    # }



In [12]:
import importlib
import graph.nodes as nodes

importlib.reload(nodes)

print(inspect.getsource(nodes.validate_review))

def validate_review(state):
    review=state.get("review")

    if not isinstance(review,dict):
        return {
            "status":"Failed",
            "error":"Review output must be a dictionary"
        }

    try:
        validate_review=CodeReview.model_validate(review)

        return {
            "review":validate_review.model_dump(),
            "status":"completed",
            "error":None
        }

    except ValidationError as e:
        return{
            "status":"failed",
            "error":f"Invalid review output: {e}"
        }

    # return {
    #     "status":"Completed"
    # }



In [13]:
invalid_review = {
    "summary": "Test review",
    "findings": [
        {
            "severity": "extreme",
            "category": "security",
            "file": "auth.js",
            "line": None,
            "title": "Test issue",
            "description": "Testing invalid severity",
            "suggested_fix": "Fix it"
        }
    ]
}

print(nodes.validate_review({"review": invalid_review}))

{'status': 'failed', 'error': "Invalid review output: 1 validation error for CodeReview\nfindings.0.severity\n  Input should be 'critical', 'high', 'medium', 'low' or 'info' [type=literal_error, input_value='extreme', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.14/v/literal_error"}


In [15]:
from main import run_code_review

result = run_code_review(
    "auth.js",
    """
+const query = "SELECT * FROM users WHERE id = " + userId;
+db.execute(query, callback);
"""
)

print(result)

{'status': 'failed', 'review': None, 'error': "Error calling model 'gemini-4-hydrogen' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-4-hydrogen, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\\nPlease retry in 16h27m7.52857062s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'de

In [16]:
import graph.nodes as nodes

# Save original function
original_review = nodes.review_code_diff

# Mock Code Review Agent
def mock_review_code_diff(filename, diff):
    return {
        "summary": "Mock code review completed successfully.",
        "findings": [
            {
                "severity": "high",
                "category": "security",
                "file": filename,
                "line": None,
                "title": "SQL Injection vulnerability",
                "description": "User input is directly concatenated into SQL query.",
                "suggested_fix": "Use parameterized SQL queries."
            }
        ]
    }

# Replace actual Gemini call temporarily
nodes.review_code_diff = mock_review_code_diff

# Run LangGraph
result = review_graph.invoke({
    "filename": "auth.js",
    "diff": "const query = 'SELECT * FROM users WHERE id=' + userId;",
    "review": None,
    "error": None,
    "status": "pending"
})

print(result)

# Restore original function
nodes.review_code_diff = original_review

{'filename': 'auth.js', 'diff': "const query = 'SELECT * FROM users WHERE id=' + userId;", 'review': {'summary': 'Mock code review completed successfully.', 'findings': [{'severity': 'high', 'category': 'security', 'file': 'auth.js', 'line': None, 'title': 'SQL Injection vulnerability', 'description': 'User input is directly concatenated into SQL query.', 'suggested_fix': 'Use parameterized SQL queries.'}]}, 'error': None, 'status': 'completed'}
